In [1]:
import math

# Standard normal CDF
def N(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

# Black-Scholes price
def black_scholes(S, K, T, r, sigma, option="call"):
    if T <= 0:
        return max(S - K, 0) if option == "call" else max(K - S, 0)

    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    if option == "call":
        return S * N(d1) - K * math.exp(-r * T) * N(d2)
    else:
        return K * math.exp(-r * T) * N(-d2) - S * N(-d1)

# Implied volatility via bisection
def implied_volatility(price, S, K, T, r, option="call",
                       tol=1e-8, max_iter=100):

    low = 1e-6
    high = 5.0          # 500% annual volatility

    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        model_price = black_scholes(S, K, T, r, mid, option)

        if abs(model_price - price) < tol:
            return mid

        if model_price < price:
            low = mid       # need higher volatility
        else:
            high = mid      # need lower volatility

    return 0.5 * (low + high)

In [2]:
S = 100        # Spot
K = 100        # Strike
T = 1.0        # 1 year
r = 0.05       # 5% risk-free
market_price = 10.45

iv = implied_volatility(market_price, S, K, T, r, "call")
print(f"Implied volatility = {iv:.4%}")

Implied volatility = 19.9984%
